# 🎨 DeepSculpt v2.0 - Complete Google Colab Guide

Welcome to DeepSculpt v2.0! This notebook provides a comprehensive guide to using our PyTorch-based 3D generative models in Google Colab.

## Features
- 🚀 **Modern PyTorch Implementation** - Latest PyTorch with GPU optimization
- 🧠 **GAN & Diffusion Models** - State-of-the-art 3D generation
- 💾 **Sparse Tensor Support** - Memory-efficient 3D processing
- 📊 **Interactive Visualizations** - Beautiful 3D plots with Plotly
- ⚡ **Colab Optimized** - Configured for Colab's constraints

## Quick Start
1. **Setup** - Run the setup cell below
2. **Generate Data** - Create synthetic 3D sculptures
3. **Train Models** - Train GAN or diffusion models
4. **Visualize** - Create beautiful 3D visualizations

---

## 🚀 Setup and Installation

Run this cell to install DeepSculpt v2.0 and check your Colab environment:

In [ ]:
# Clone DeepSculpt repository
!git clone https://github.com/deepsculpt/deepsculpt.git
%cd deepsculpt/deepSculpt/deepsculpt

# Run Colab setup script
!python colab_setup.py

# Import essential modules
import sys
import os
sys.path.append('.')

import torch
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
import json
import yaml
from tqdm.auto import tqdm

# Check system info
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🎯 CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Running on CPU - training will be slower")

print("\n✅ Setup complete! Ready to start sculpting!")

## ⚙️ Configuration

Load Colab-optimized configuration:

In [ ]:
# Load Colab configuration
with open('config_colab.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Display key settings
print("🔧 Colab Configuration:")
print(f"  Model: {config['model']['model_type']} (void_dim: {config['model']['void_dim']})")
print(f"  Training: {config['training']['epochs']} epochs, batch size {config['training']['batch_size']}")
print(f"  Data: {config['data']['num_samples']} samples, {config['data']['num_shapes']} shapes")
print(f"  Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"  Mixed Precision: {config['training']['mixed_precision'] and torch.cuda.is_available()}")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🎯 Using device: {device}")

## 📊 Data Generation

Generate synthetic 3D sculpture data:

In [ ]:
# Generate synthetic data using the command line interface
!python main.py generate-data \
    --num-samples=50 \
    --void-dim=32 \
    --num-shapes=3 \
    --sparse \
    --output-dir=./colab_data

# Check generated data
data_files = list(Path('./colab_data').glob('*.pt'))
print(f"\n📁 Generated {len(data_files)} data files")

if data_files:
    # Load and inspect a sample
    sample = torch.load(data_files[0])
    print(f"📏 Sample shape: {sample['structure'].shape}")
    print(f"🎨 Colors shape: {sample['colors'].shape}")
    print(f"💾 Sparsity: {(sample['structure'] == 0).float().mean():.3f}")
    
    print("\n✅ Data generation complete!")
else:
    print("❌ No data files generated")

## 🎨 Data Visualization

Visualize the generated 3D sculptures:

In [ ]:
# Import visualization modules
from core.visualization.pytorch_visualization import PyTorchVisualizer

# Create visualizer
visualizer = PyTorchVisualizer(backend='plotly', device=device)

if data_files:
    # Load a sample for visualization
    sample = torch.load(data_files[0])
    structure = sample['structure']
    colors = sample['colors']
    
    print(f"🎨 Visualizing sculpture with shape: {structure.shape}")
    
    # Create 3D visualization
    try:
        # Plot the sculpture
        fig = visualizer.plot_sculpture(structure, colors)
        
        if fig:
            fig.show()
            print("✅ 3D visualization created!")
        else:
            print("⚠️ Visualization created but not displayed")
            
    except Exception as e:
        print(f"❌ Visualization error: {e}")
        
        # Fallback: show basic statistics
        print("\n📊 Sculpture Statistics:")
        print(f"  Non-zero voxels: {(structure > 0).sum().item()}")
        print(f"  Unique colors: {colors.unique().numel()}")
        print(f"  Value range: [{structure.min():.3f}, {structure.max():.3f}]")
else:
    print("❌ No data files to visualize")

## 🧠 GAN Training

Train a Generative Adversarial Network for 3D sculpture generation:

In [ ]:
# Train GAN model with Colab-optimized settings
!python main.py train-gan \
    --model-type=simple \
    --epochs=5 \
    --batch-size=4 \
    --void-dim=32 \
    --learning-rate=0.0002 \
    --mixed-precision \
    --generate-samples \
    --output-dir=./colab_gan_results \
    --verbose

print("\n🎉 GAN training complete!")

# Check results
results_dir = Path('./colab_gan_results')
if results_dir.exists():
    result_files = list(results_dir.rglob('*'))
    print(f"📁 Generated {len(result_files)} result files")
    
    # Look for model files
    model_files = list(results_dir.glob('**/generator_final.pt'))
    if model_files:
        print(f"🤖 Trained model saved: {model_files[0]}")
    
    # Look for sample images
    sample_images = list(results_dir.glob('**/samples/*.png'))
    if sample_images:
        print(f"🖼️ Generated {len(sample_images)} sample visualizations")
else:
    print("❌ No training results found")

## 🎲 Generate Samples from Trained GAN

Generate new sculptures using the trained GAN model:

In [ ]:
# Find the trained generator model
model_files = list(Path('./colab_gan_results').glob('**/generator_final.pt'))

if model_files:
    model_path = model_files[0]
    print(f"🤖 Using model: {model_path}")
    
    # Generate samples
    !python main.py sample-gan \
        --checkpoint={model_path} \
        --num-samples=5 \
        --visualize \
        --output-dir=./colab_samples
    
    # Check generated samples
    sample_files = list(Path('./colab_samples').glob('*.pt'))
    sample_images = list(Path('./colab_samples').glob('*.png'))
    
    print(f"\n🎨 Generated {len(sample_files)} samples")
    print(f"🖼️ Created {len(sample_images)} visualizations")
    
    # Display a sample if available
    if sample_files:
        sample = torch.load(sample_files[0])
        print(f"\n📏 Sample shape: {sample.shape}")
        print(f"📊 Value range: [{sample.min():.3f}, {sample.max():.3f}]")
        print(f"💾 Sparsity: {(sample == 0).float().mean():.3f}")
        
        # Try to visualize
        try:
            fig = visualizer.plot_sculpture(sample.squeeze())
            if fig:
                fig.show()
                print("✅ Generated sample visualization!")
        except Exception as e:
            print(f"⚠️ Visualization error: {e}")
    
    print("\n🎉 Sample generation complete!")
else:
    print("❌ No trained model found. Please run GAN training first.")

## 🌊 Diffusion Model Training

Train a diffusion model for high-quality 3D generation:

In [ ]:
# Train diffusion model
!python main.py train-diffusion \
    --epochs=5 \
    --batch-size=2 \
    --void-dim=32 \
    --timesteps=100 \
    --noise-schedule=linear \
    --learning-rate=1e-4 \
    --mixed-precision \
    --output-dir=./colab_diffusion_results

print("\n🌊 Diffusion training complete!")

# Check results
diffusion_dir = Path('./colab_diffusion_results')
if diffusion_dir.exists():
    result_files = list(diffusion_dir.rglob('*'))
    print(f"📁 Generated {len(result_files)} result files")
    
    # Look for model files
    model_files = list(diffusion_dir.glob('**/diffusion_final.pt'))
    if model_files:
        print(f"🤖 Trained diffusion model: {model_files[0]}")
else:
    print("❌ No diffusion training results found")

## 🎭 Generate Samples from Diffusion Model

Generate high-quality samples using the trained diffusion model:

In [ ]:
# Find the trained diffusion model
diffusion_models = list(Path('./colab_diffusion_results').glob('**/diffusion_final.pt'))

if diffusion_models:
    model_path = diffusion_models[0]
    print(f"🌊 Using diffusion model: {model_path}")
    
    # Generate samples with diffusion
    !python main.py sample-diffusion \
        --checkpoint={model_path} \
        --num-samples=3 \
        --num-steps=20 \
        --visualize \
        --output-dir=./colab_diffusion_samples
    
    # Check generated samples
    sample_files = list(Path('./colab_diffusion_samples').glob('*.pt'))
    sample_images = list(Path('./colab_diffusion_samples').glob('*.png'))
    
    print(f"\n🎨 Generated {len(sample_files)} diffusion samples")
    print(f"🖼️ Created {len(sample_images)} visualizations")
    
    # Display a sample if available
    if sample_files:
        sample = torch.load(sample_files[0])
        print(f"\n📏 Diffusion sample shape: {sample.shape}")
        print(f"📊 Value range: [{sample.min():.3f}, {sample.max():.3f}]")
        
        # Try to visualize
        try:
            fig = visualizer.plot_sculpture(sample.squeeze())
            if fig:
                fig.show()
                print("✅ Diffusion sample visualization!")
        except Exception as e:
            print(f"⚠️ Visualization error: {e}")
    
    print("\n🎉 Diffusion sampling complete!")
else:
    print("❌ No trained diffusion model found. Please run diffusion training first.")

## ⚡ Performance Benchmarking

Test the performance of your Colab setup:

In [ ]:
# Run performance benchmarks
!python main.py benchmark \
    --model-type=simple \
    --batch-size=4 \
    --void-dim=32 \
    --profile-memory \
    --save-results \
    --output-dir=./colab_benchmarks

# Check benchmark results
benchmark_files = list(Path('./colab_benchmarks').glob('*.json'))
if benchmark_files:
    with open(benchmark_files[0], 'r') as f:
        results = json.load(f)
    
    print("\n⚡ Benchmark Results:")
    print("=" * 40)
    for key, value in results.items():
        if isinstance(value, float):
            print(f"{key:25}: {value:.4f}")
        else:
            print(f"{key:25}: {value}")
    
    # Performance assessment
    if 'throughput_samples_per_sec' in results:
        throughput = results['throughput_samples_per_sec']
        if throughput > 10:
            print("\n🚀 Excellent performance!")
        elif throughput > 5:
            print("\n✅ Good performance")
        else:
            print("\n⚠️ Limited performance - consider using smaller models")
else:
    print("❌ No benchmark results found")

print("\n🎯 Benchmarking complete!")

## 💡 Tips for Google Colab

### Memory Management
- Use smaller `void_dim` (16-32) for limited memory
- Enable `sparse` tensors to save memory
- Reduce `batch_size` if you get OOM errors
- Use `mixed_precision` training when available

### Training Tips
- Start with fewer `epochs` (5-10) for quick testing
- Use `simple` model type for faster training
- Save checkpoints frequently in case of disconnection
- Monitor GPU usage to avoid hitting limits

### Colab Limitations
- Sessions timeout after 12 hours (free) or 24 hours (Pro)
- GPU access may be limited during peak times
- Disk space is limited (~100GB)
- Network access may be restricted

### Optimization
- Use Google Drive to save important results
- Compress large files to save space
- Clear outputs regularly to reduce notebook size
- Use the Colab-optimized configuration file

---

## 🧹 Cleanup and Save Results

Clean up temporary files and save important results:

In [ ]:
import shutil

# Create archive of important results
print("📦 Creating results archive...")

# List of important directories to archive
important_dirs = [
    './colab_gan_results',
    './colab_diffusion_results', 
    './colab_samples',
    './colab_diffusion_samples',
    './colab_benchmarks'
]

# Check what exists
existing_dirs = [d for d in important_dirs if Path(d).exists()]
print(f"📁 Found {len(existing_dirs)} result directories")

if existing_dirs:
    # Create archive
    archive_name = 'deepsculpt_colab_results'
    shutil.make_archive(archive_name, 'zip', '.', 
                       base_dir=None)
    
    archive_path = f"{archive_name}.zip"
    if Path(archive_path).exists():
        size_mb = Path(archive_path).stat().st_size / 1024 / 1024
        print(f"✅ Archive created: {archive_path} ({size_mb:.1f} MB)")
        
        # Instructions for downloading
        print("\n📥 To download results:")
        print(f"   files.download('{archive_path}')")
    else:
        print("❌ Failed to create archive")
else:
    print("⚠️ No results to archive")

# Show disk usage
try:
    total, used, free = shutil.disk_usage('.')
    print(f"\n💾 Disk Usage:")
    print(f"   Total: {total / 1e9:.1f} GB")
    print(f"   Used:  {used / 1e9:.1f} GB")
    print(f"   Free:  {free / 1e9:.1f} GB")
except:
    print("⚠️ Could not determine disk usage")

print("\n🎉 Session complete! Thank you for using DeepSculpt v2.0!")

---

## 🎨 DeepSculpt v2.0

**Modern PyTorch-based 3D Generative Models**

- 🌐 **Website**: [deepsculpt.ai](https://deepsculpt.ai)
- 📚 **Documentation**: [docs.deepsculpt.ai](https://docs.deepsculpt.ai)
- 🐙 **GitHub**: [github.com/deepsculpt/deepsculpt](https://github.com/deepsculpt/deepsculpt)
- 💬 **Discord**: [discord.gg/deepsculpt](https://discord.gg/deepsculpt)

### Support
- 📧 Email: support@deepsculpt.ai
- 🐛 Issues: [GitHub Issues](https://github.com/deepsculpt/deepsculpt/issues)
- 💡 Feature Requests: [GitHub Discussions](https://github.com/deepsculpt/deepsculpt/discussions)

**Happy Sculpting!** 🎨✨

---